[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_9/03_tag_9_instanzsegmentierung_mask_rcnn.ipynb)

# Tag 9 – 03: Instanzsegmentierung mit Mask R-CNN

Wir verbinden jetzt **Objekterkennung** und **Segmentierung**: Das Modell findet jede Person als eigene Instanz, gibt eine Bounding Box aus und markiert zusätzlich **jeden zugehörigen Pixel**. Dafür verwenden wir `Mask R-CNN`, ein praxiserprobtes Standardnetz aus `torchvision`.

Führe zuerst `00_tag_9_datensatz_vorbereiten.ipynb` aus. Es lädt weiterhin unverändert den Penn-Fudan-Datensatz herunter. Dessen Originalmasken enthalten bereits für jede Person eine eigene ID – genau das Label, das Mask R-CNN benötigt.

## Lernziel: Input, Target und Ausgabe

| Rolle | Form / Beispiel | Bedeutung |
| --- | --- | --- |
| Input `image` | `(3, H, W)` | RGB-Foto mit Originalgröße und Werten von `0` bis `1` |
| Target `boxes` | `(N, 4)` | `N` echte Personenboxen im Format `[x_min, y_min, x_max, y_max]` |
| Target `labels` | `(N,)` | für jede Instanz `1` für `Person`; `0` reserviert das Modell für Hintergrund |
| Target `masks` | `(N, H, W)` | `N` getrennte binäre Masken – eine pro Person |
| Ausgabe | `boxes`, `labels`, `scores`, `masks` | eine unterschiedlich lange Liste gefundener Personen |

`H` und `W` dürfen pro Bild verschieden sein. Deshalb bildet unser `DataLoader` keinen großen Bildtensor; Mask R-CNN bekommt eine Python-Liste einzelner Bilder und skaliert sie intern zusammen mit den Targets.

## Abgrenzung: semantisch oder Instanzen?

| Aufgabe | Ergebnis bei zwei Personen |
| --- | --- |
| Objekterkennung | zwei Bounding Boxes |
| Semantische Segmentierung (Notebook 02) | eine gemeinsame Fläche `Person` |
| **Instanzsegmentierung (dieses Notebook)** | zwei Boxes **und zwei getrennte Masken** |

Die getrennten Masken sind entscheidend, wenn Personen einander überdecken. Die IDs `1`, `2`, … in einer Penn-Fudan-Maske werden deshalb **nicht** zu einer einzigen binären Personenmaske zusammengefasst.

Eine Box beantwortet nur: *Wo ungefähr ist ein Objekt?* Eine Instanzmaske beantwortet zusätzlich: *Welche Pixel gehören genau zu diesem Objekt?* Aus jeder Instanzmaske lässt sich eine Box ableiten; umgekehrt kann eine Box aber niemals die genaue Silhouette rekonstruieren.

In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights, maskrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms import functional as TF

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

DATASET_ROOT = Path('datasets/PennFudanPed')
BATCH_SIZE = 2 if torch.cuda.is_available() else 1
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Gerät:', DEVICE)
print('Batch-Größe:', BATCH_SIZE)
print('Datensatz:', DATASET_ROOT.resolve())

## Dataset: eine Maske pro Person erzeugen

`Mask R-CNN` erwartet pro Bild ein Wörterbuch mit `boxes`, `labels` und `masks`. Alle drei Einträge haben eine erste Dimension `N`: die Anzahl der Personen im Bild. Aus der originalen ID-Maske wird für jede positive ID eine boolesche Einzelmaske. Die zugehörige Bounding Box berechnen wir direkt aus ihren äußersten Pixeln.

Die Bilder behalten zunächst ihre Originalgröße. Das Standardmodell skaliert Bild und Targets intern gemeinsam auf eine passende Größe. Beim horizontalen Spiegeln berechnen wir die Boxes anschließend neu; so können Bild, Maske und Box nicht auseinanderlaufen.

In [ ]:
def boxes_from_masks(masks):
    """Berechnet [x_min, y_min, x_max, y_max] für jede Einzelmaske."""
    boxes = []
    for mask in masks:
        # torch.where liefert erst y (Zeile), dann x (Spalte).
        y_coordinates, x_coordinates = torch.where(mask)
        boxes.append(torch.tensor([
            x_coordinates.min(), y_coordinates.min(),
            x_coordinates.max() + 1, y_coordinates.max() + 1,
        ], dtype=torch.float32))
    return torch.stack(boxes)


class PennFudanInstanceDataset(Dataset):
    def __init__(self, split, augment=False):
        annotation_path = DATASET_ROOT / f'segmentation_{split}.json'
        if not annotation_path.exists():
            raise FileNotFoundError(
                f'{annotation_path} fehlt. Führe zuerst Notebook 00 aus.'
            )
        self.samples = json.loads(annotation_path.read_text(encoding='utf-8'))
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = Image.open(DATASET_ROOT / sample['image']).convert('RGB')
        # Die PNG-Maske enthält 0 für Hintergrund und 1, 2, ... für Personen.
        instance_ids = np.asarray(Image.open(DATASET_ROOT / sample['mask']))

        person_ids = np.unique(instance_ids)
        person_ids = person_ids[person_ids != 0]  # 0 bedeutet Hintergrund
        # Broadcasting erzeugt eine Ebene pro ID: (N, H, W), noch keine Vereinigung.
        masks = torch.from_numpy(
            (instance_ids == person_ids[:, None, None]).astype(np.bool_)
        )

        if self.augment and random.random() < 0.5:
            # Geometrische Transformationen müssen Bild UND jede Zielmaske verändern.
            image = TF.hflip(image)
            masks = torch.flip(masks, dims=[2])

        # Nach einer Spiegelung werden die Boxes aus den Masken neu berechnet.
        boxes = boxes_from_masks(masks)
        labels = torch.ones((len(person_ids),), dtype=torch.int64)
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        target = {
            'boxes': boxes,
            'labels': labels,          # 1 = Person; 0 ist im Modell Hintergrund
            'masks': masks.to(torch.uint8),
            'image_id': torch.tensor([index]),
            'area': area,              # Fläche der Box; Teil des üblichen Detection-Target-Formats
            'iscrowd': torch.zeros((len(person_ids),), dtype=torch.int64),
        }
        return TF.to_tensor(image), target


def collate_instances(batch):
    # Bilder haben unterschiedliche Höhen und Breiten; deshalb keine Stapelung hier.
    # Mask R-CNN verarbeitet die Listen und führt seine gemeinsame Skalierung selbst aus.
    return tuple(zip(*batch))


train_dataset = PennFudanInstanceDataset('train', augment=True)
valid_dataset = PennFudanInstanceDataset('valid')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_instances)
valid_loader = DataLoader(valid_dataset, batch_size=1, shuffle=False,
                          num_workers=0, collate_fn=collate_instances)

image, target = valid_dataset[0]
print(f'Training: {len(train_dataset)} Bilder; Validierung: {len(valid_dataset)} Bilder')
print('Bild-Shape:', tuple(image.shape))
print('Personen im ersten Validierungsbild:', len(target['labels']))
print('Boxes:', target['boxes'].tolist())
print('Masken-Shape:', tuple(target['masks'].shape))
assert target['masks'].shape[0] == len(target['labels']) == len(target['boxes'])
assert target['masks'].shape[1:] == image.shape[1:]
assert torch.all(target['boxes'][:, 2] > target['boxes'][:, 0])
assert torch.all(target['boxes'][:, 3] > target['boxes'][:, 1])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image.permute(1, 2, 0))
for person_index, box in enumerate(target['boxes']):
    x0, y0, x1, y1 = box.tolist()
    axes[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                      fill=False, edgecolor='yellow', linewidth=2))
    axes[0].text(x0, max(4, y0 - 4), f'Person {person_index + 1}', color='yellow',
                 bbox={'facecolor': 'black', 'alpha': 0.55, 'pad': 1})
axes[0].set_title('Ein Bild – mehrere ermittelte Boxes')

instance_view = np.zeros((*target['masks'].shape[1:], 4), dtype=np.float32)
colors = plt.cm.tab10(np.arange(len(target['masks'])))
for person_index, mask in enumerate(target['masks']):
    instance_view[mask.bool().numpy()] = colors[person_index]
axes[1].imshow(instance_view)
axes[1].set_title('Ein Pixel gehört zu genau einer Personen-Instanz')
for axis in axes:
    axis.axis('off')
plt.tight_layout()

## Das Standardnetz: Mask R-CNN mit ResNet-50 und FPN

Mask R-CNN erweitert Faster R-CNN um einen Masken-Zweig. Der Datenfluss ist:

```text
RGB-Bild
  │  interne Normalisierung und gemeinsame Skalierung von Bild, Boxes und Masken
  ▼
ResNet-50-Backbone ──► FPN: Merkmale auf mehreren Größenstufen
  │
  ▼
RPN (Region Proposal Network): viele Anker → wenige interessante Regionsvorschläge
  │                              
  ▼
RoI Align: für jede Region ein gleich großes Merkmalsraster ohne grobes Runden
  ├── Box-Head: Klasse + präzisere Bounding Box
  └── Mask-Head: eigene Pixelmaske für dieselbe Region
```

| Baustein | Was er lernt | Warum er nötig ist |
| --- | --- | --- |
| **ResNet-50** | Kanten, Texturen, Körperteile und abstraktere Objektmerkmale | Aus Rohpixeln werden aussagekräftige Bildmerkmale. |
| **FPN** | dieselben Merkmale auf groben und feinen Auflösungen | Kleine und große Personen brauchen unterschiedlich große Sichtfelder. |
| **RPN** | welche Anker wahrscheinlich ein Objekt enthalten und wie sie verschoben werden | Sucht effizient mögliche Personenregionen, statt jede mögliche Box vollständig auszuwerten. |
| **RoI Align** | keine neuen Gewichte, sondern präzises Abtasten | Jede unterschiedlich große Region wird in ein gleich großes Merkmalsraster überführt; die Boxgrenzen bleiben dabei genauer als bei RoI Pooling. |
| **Box-Head** | Hintergrund/Person und vier Boxkorrekturen | Prüft und verfeinert die RPN-Vorschläge. |
| **Mask-Head** | eine räumliche Binärmaske pro positiver Region | Trennt die exakten Personenpixel vom Hintergrund innerhalb der Box. |

### Was heißt FPN genau?

**FPN** steht für **Feature Pyramid Network**, also ein *Pyramidennetz aus Merkmalen*. ResNet-50 erzeugt nach mehreren Blöcken Merkmalskarten mit unterschiedlicher Auflösung: Frühe Karten sind groß und enthalten feine Ortsinformation, tiefe Karten sind klein und enthalten mehr Bedeutung wie „das sieht nach einem Menschen aus“. Nur die letzte, grobe Karte wäre für kleine Personen ungünstig.

FPN baut deshalb von grob nach fein eine zweite Pyramide: Eine tiefe Karte wird hochskaliert, über eine seitliche 1×1-Transformation mit der Karte der nächstfeineren ResNet-Stufe addiert und anschließend geglättet. So entstehen die FPN-Ebenen `P2`, `P3`, `P4`, `P5`: Jede enthält starke semantische Information **und** eine passende Ortsauflösung. Das RPN und RoI Align wählen für kleine Regionsvorschläge eher eine feine Ebene wie `P2`, für große Regionen eher eine grobe Ebene wie `P5`.

### Was passiert nach der Box? – kein zweites Bild, kein Ausschneiden

Es gibt **kein** Schema `Bild → Box → außerhalb schwärzen → zweites Segmentierungsmodell`. Stattdessen arbeiten Box- und Mask-Zweig mit denselben bereits berechneten FPN-Merkmalskarten. Die originale Bildmatrix wird nach dem Backbone nicht erneut in ein Netz gesteckt.

1. Das RPN erzeugt Vorschlagsboxes aus den FPN-Karten.
2. `box_roi_pool` führt für jede Vorschlagsbox **RoI Align** auf den FPN-Karten aus. Das Ergebnis ist ein kleines, immer gleich großes Merkmalsraster – kein Bildausschnitt. Der Box-Head entscheidet Person/Hintergrund und verschiebt die Box genauer.
3. Im Inferenzmodus nimmt `torchvision` die danach übrig gebliebenen, verfeinerten Detektionsboxes. `mask_roi_pool` führt für diese Boxes ein zweites RoI Align auf **denselben FPN-Karten** aus.
4. Der Mask-Head erzeugt pro Box eine kleine Wahrscheinlichkeitsmaske in lokalen Box-Koordinaten. Diese wird auf die Größe der finalen Box skaliert und an deren Position auf eine Ausgabefläche gelegt. Außerhalb der Box gibt es keine positive Instanzmaske – nicht weil Bildpixel geschwärzt wurden, sondern weil die lokale Maske dort nicht platziert wird.

Im **Training** gibt es eine wichtige Abweichung: Der Mask-Head erhält die positiven Regionsvorschläge, die per IoU einer echten Person zugeordnet wurden. Die passende echte Instanzmaske wird auf dieselbe lokale Rastergröße zugeschnitten und skaliert. So wird der Masken-Loss pro Person und pro Region berechnet, auch bevor die endgültige Box bereits perfekt ist.

Box-Head und Mask-Head sind somit zwei spezialisierte Zweige eines einzigen Mask-R-CNN-Modells: Sie teilen Backbone, FPN und die Information *wo* eine relevante Region liegt, haben aber getrennte lernbare Ausgaben. Dadurch kann die Box präzise lokalisiert und die Silhouette unabhängig davon fein geformt werden.

Die Maske wird also **nicht** für das ganze Bild in einem Schritt erzeugt. Erst wird eine Person lokalisiert; dann segmentiert der Mask-Head genau diese eine Region. Dadurch bleiben zwei überlappende Personen getrennte Instanzen. NMS im Detektionszweig reduziert anschließend überlappende Doppelvorschläge.

Wir starten mit COCO-vortrainierten Gewichten. Der Datensatzdownload aus Notebook 00 bleibt davon getrennt: Beim ersten Start dieser Zelle werden nur die Modellgewichte in den lokalen PyTorch-Cache geladen. Danach ersetzen wir die beiden Ausgabeköpfe für unsere Klassen `Hintergrund` und `Person`.

In [ ]:
USE_PRETRAINED_COCO = True
NUM_CLASSES = 2  # Hintergrund (0) und Person (1)

# Die vollständigen COCO-Gewichte enthalten Backbone, RPN, Box- und Mask-Head.
weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if USE_PRETRAINED_COCO else None
model = maskrcnn_resnet50_fpn_v2(
    weights=weights,
    weights_backbone=None,  # bei USE_PRETRAINED_COCO=False auch keinen zweiten Gewichte-Download starten
    min_size=480,  # kleinere Kante: auf CPU etwas schneller als das Standardlimit 800
    max_size=800,
)

# COCO kennt viele Klassen. Wir ersetzen nur seine letzte Klassifikations- und
# Box-Ausgabe durch einen neuen Kopf mit genau zwei Klassen (0 Hintergrund, 1 Person).
box_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(box_features, NUM_CLASSES)

# Auch der Mask-Head erhält einen neuen letzten Teil: Er gibt für jede positive
# Region einen Masken-Logit pro Klasse aus. Für uns zählt nur Kanal 1 = Person.
mask_features = model.roi_heads.mask_predictor.conv5_mask.in_channels
model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, 256, NUM_CLASSES)

# Zunächst lernt wirklich nur der neue Vorhersageteil. Das Einfrieren des
# gesamten Modells umfasst auch RPN, FPN und die vortrainierten Zwischen-Head-
# Schichten; anschließend öffnen wir ausschließlich die zwei neuen Predictor.
for parameter in model.parameters():
    parameter.requires_grad = False
for head in (model.roi_heads.box_predictor, model.roi_heads.mask_predictor):
    for parameter in head.parameters():
        parameter.requires_grad = True

model = model.to(DEVICE)
trainable_parameters = sum(parameter.numel() for parameter in model.parameters()
                           if parameter.requires_grad)
print(f'Trainierbare Parameter im Warm-up: {trainable_parameters:,}')
print('\nNeuer Box-Head:\n', model.roi_heads.box_predictor)
print('\nNeuer Mask-Head:\n', model.roi_heads.mask_predictor)

### Die Bausteine im tatsächlichen `torchvision`-Modell

Die eine Variable `model` enthält alle zuvor beschriebenen Teile – es werden nicht zwei Modelle erzeugt oder nacheinander mit Bilddaten gefüttert. Die Namen im folgenden Code entsprechen dem internen Ablauf von `torchvision`:

```python
features = model.backbone(...)                    # ResNet-50 + FPN, einmal pro Bild
proposals, _ = model.rpn(..., features, ...)      # RPN arbeitet auf diesen features
box_features = model.roi_heads.box_roi_pool(features, proposals, ...)
# Box-Head → Klassen und verfeinerte Boxes
mask_features = model.roi_heads.mask_roi_pool(features, final_boxes, ...)
# Mask-Head → lokale Masken für genau diese final_boxes
```

Das ist vereinfachter, kommentierter Pseudocode des internen Vorwärtsdurchlaufs. Die nächste Zelle zeigt die echten Module und ihre RoI-Align-Ausgabegrößen. `box_roi_pool` und `mask_roi_pool` erhalten beide FPN-Merkmale, aber unterschiedliche Rastergrößen, weil Box-Klassifikation und Pixelmaske verschiedene Detailgrade benötigen.

In [ ]:
# Diese Ausgaben machen die Modulnamen aus dem Ablauf oben im echten Modell sichtbar.
print('Backbone mit FPN:', type(model.backbone).__name__)
print('Ankergrößen pro FPN-Ebene:', model.rpn.anchor_generator.sizes)
print('Box-RoI-Align:', model.roi_heads.box_roi_pool)
print('  Ausgabegröße pro Vorschlag:', model.roi_heads.box_roi_pool.output_size)
print('Mask-RoI-Align:', model.roi_heads.mask_roi_pool)
print('  Ausgabegröße pro finaler Detektion:', model.roi_heads.mask_roi_pool.output_size)

# Beide Zweige greifen auf FPN-Feature-Maps zu, nicht auf einen ausgeschnittenen RGB-Tensor.
assert model.roi_heads.box_roi_pool.featmap_names == model.roi_heads.mask_roi_pool.featmap_names
print('Geteilte FPN-Ebenen:', model.roi_heads.box_roi_pool.featmap_names)

# `model([image])` ruft Backbone, RPN und beide RoI-Zweige in genau dieser
# Reihenfolge intern auf. Der Mask-Head erhält dabei FPN-Merkmale plus die
# finalen Box-Koordinaten über mask_roi_pool – kein zweites RGB-Bild.

## Die Verbindung von Box-Head und Mask-Head – Schritt für Schritt

### Inferenz: ein Bild vorhersagen

Bei einer Vorhersage gibt es **keine echte Box** und damit auch keine IoU mit einer echten Box. Das Bild läuft genau **einmal** durch ResNet-50 und FPN. Danach werden nur kleine Merkmalsraster verarbeitet, nicht nochmals das vollständige Bild.

```text
1. Bild ──einmal──► Backbone + FPN ──► gemeinsame FPN-Merkmalskarten
2. RPN nutzt diese Karten ──► viele grobe Vorschlagsboxes
3. Box-RoI-Align nutzt FPN + Vorschlagsboxes ──► ein Merkmalsraster je Vorschlag
4. Box-Head ──► Klassenscore und verfeinerte Box je Vorschlag
5. Score-Schwelle + NMS ──► finale Detektionsboxes
6. Mask-RoI-Align nutzt FPN + FINALE Detektionsboxes ──► ein Merkmalsraster je finaler Box
7. Mask-Head ──► eine lokale Personenmaske je finaler Box
```

Schritt 6 ist tatsächlich ein **zweiter kleiner RoI-Align-/Mask-Head-Durchlauf**, aber kein zweiter Durchlauf des Bildes durch ResNet oder FPN. Hat das Modell nach Score-Schwelle und NMS drei Personen gefunden, erhält der Mask-Head genau drei RoI-Merkmalsraster. Bei keiner Detektion erhält er keine Region und erzeugt keine Maske.

Der Mask-Head bekommt vom Box-Zweig **die finalen Box-Koordinaten** als Ortsinformation: Sie bestimmen, welchen Ausschnitt der FPN-Karten `mask_roi_pool` abtastet. Er bekommt normalerweise nicht den Klassenlogit oder den flachen Feature-Vektor des Box-Heads als zusätzlichen Eingabetensor. Beide Heads teilen die FPN-Karten; die Box verbindet sie über ihre Geometrie. Das vorhergesagte Klassenlabel wählt anschließend den passenden Maskenkanal aus. Hier gibt es nur den Kanal `Person`.

### Wie Box und FPN-Tensor konkret zusammenpassen: RoI Align

Die Box wird **nicht** mit den FPN-Kanälen verkettet. Sie ist eine geometrische Anweisung: „Nimm aus dieser FPN-Karte genau das Rechteck an dieser Stelle.“ `mask_roi_pool` wählt zuerst anhand der Boxgröße eine passende FPN-Stufe und rechnet die Boxkoordinaten auf deren Auflösung um. Anschließend legt RoI Align ein festes Abtastraster über das Rechteck und liest die Werte an diesen Punkten per bilinearer Interpolation.

Beispiel ohne zusätzliche Bildskalierung: Ein Bild ist `800 × 600` Pixel groß. Eine feine FPN-Karte `P2` hat bei einem Schritt von 4 Pixeln die Größe `200 × 150`. Die Box `[100, 80, 300, 480]` im Bild entspricht auf `P2` dem Rechteck `[25, 20, 75, 120]`. RoI Align teilt dieses Rechteck **nicht** in gleich große Pixelblöcke, sondern setzt darin ein festes 14×14-Abtastraster. So wird daraus immer ein Tensor mit Form `256 × 14 × 14` – egal ob die ursprüngliche Box 200×400, 60×100 oder 500×500 Pixel groß war.

Bei mehreren finalen Detektionen wird nur eine neue erste Dimension ergänzt: Aus `M` Boxes wird `mask_roi_pool` daher ein Tensor `(M, 256, 14, 14)`. Genau deshalb dürfen die Bildausschnitte ursprünglich unterschiedlich groß sein, obwohl der Mask-Head eine feste Eingabeform benötigt. Der Mask-Head macht daraus pro Klasse ein `28 × 28`-Maskenlogit. Dieses lokale 28×28-Ergebnis wird auf die Größe der jeweiligen finalen Box skaliert und im Bild positioniert.

### Wofür IoU dann wirklich verwendet wird

| Situation | Wozu dient IoU? | Kennt das Modell die echte Box? |
| --- | --- | --- |
| **Training** | Zuordnung: Ist ein Anker/Vorschlag positiv, negativ oder zu ignorieren? Welche echte Box und welche echte Maske ist sein Target? | Ja, sie steht im Trainings-Target. |
| **Inferenz: NMS** | Vergleich **vorhergesagter** Boxes miteinander. Stark überlappende Doppelungen werden entfernt. | Nein. |
| **Inferenz: Auswahl einer Person** | Score-Schwelle und NMS bestimmen, welche Boxes final bleiben. | Nein. |
| **Evaluation** | Vergleich vorhergesagter mit echten Boxes/Masken, um IoU oder mAP zu messen. | Ja, aber nur zur Bewertung, nicht als Modellinput. |

Die Formulierung „valide Box“ bedeutet bei der Inferenz daher: hohe genug vorhergesagte Person-Wahrscheinlichkeit und kein von NMS unterdrücktes Duplikat – **nicht** „hat genug IoU mit einer unbekannten echten Box“.

In [ ]:
# Optionaler Blick in den echten Inferenzdurchlauf. Diese Zelle führt nicht
# noch einmal den Backbone aus: `features` wird einmal erzeugt und danach von
# RPN, Box-RoI-Align und Mask-RoI-Align wiederverwendet.
@torch.no_grad()
def inspect_roi_align_shapes(model, one_image):
    was_training = model.training
    model.eval()

    # Dasselbe Vorverarbeiten wie im normalen model([image])-Aufruf.
    images, _ = model.transform([one_image.to(DEVICE)], None)
    features = model.backbone(images.tensors)
    print('FPN-Karten (Batch, Kanäle, Höhe, Breite):')
    for level, feature_map in features.items():
        print(f'  Ebene {level}: {tuple(feature_map.shape)}')

    # RPN und RoI-Heads verwenden exakt dieses Dictionary `features`.
    proposals, _ = model.rpn(images, features, None)
    detections, _ = model.roi_heads(features, proposals, images.image_sizes, None)
    final_boxes = [detection['boxes'] for detection in detections]
    # roi_heads hat den Mask-Zweig intern bereits ausgeführt. Die folgenden
    # drei Zeilen wiederholen nur dessen Maskenteil, um seine Tensorformen
    # sichtbar zu machen – weiterhin ohne einen zweiten Backbone-Durchlauf.
    number_of_boxes = sum(len(boxes) for boxes in final_boxes)
    print('Finale Boxes nach Box-Head/NMS:', number_of_boxes)

    if number_of_boxes:
        # Hier werden Box-Koordinaten zu festen 14×14-Merkmalsrastern: keine
        # Pixel des RGB-Bildes, sondern abgetastete Werte aus `features`.
        mask_roi_features = model.roi_heads.mask_roi_pool(
            features, final_boxes, images.image_sizes
        )
        mask_logits = model.roi_heads.mask_predictor(
            model.roi_heads.mask_head(mask_roi_features)
        )
        print('Input des Mask-Heads (M, 256, 14, 14):', tuple(mask_roi_features.shape))
        print('Maskenlogits (M, Klassen, 28, 28):', tuple(mask_logits.shape))
    else:
        print('Keine finale Box: daher erhält der Mask-Head keine Region.')

    model.train(was_training)


# Vor dem Training können die Köpfe noch zufällig sein; die Tensorformen und
# der Verbindungsmechanismus sind trotzdem schon dieselben.
inspect_roi_align_shapes(model, valid_dataset[0][0])

## Loss: fünf Teilaufgaben, ein gemeinsamer Rückwärtsdurchlauf

Im Trainingsmodus liefert `Mask R-CNN` kein einzelnes Logit-Tensor zurück, sondern ein Wörterbuch aus Teilverlusten. `torchvision` ordnet Anker und Regionsvorschläge anhand ihrer Überlappung (IoU) den echten Boxes zu; positive Regionen erhalten zusätzlich die passende Instanzmaske. Der Gesamt-Loss ist die Summe:

$$L = L_{rpn\_objectness} + L_{rpn\_box} + L_{classifier} + L_{box\_reg} + L_{mask}.$$

| Schlüssel aus `torchvision` | Lernaufgabe | Typ des Fehlers |
| --- | --- | --- |
| `loss_objectness` | Enthält ein RPN-Anker überhaupt eine Person? | binäre Kreuzentropie über Anker |
| `loss_rpn_box_reg` | Wie muss ein positiver Anker verschoben und skaliert werden? | robuste Box-Regression |
| `loss_classifier` | Ist ein Regionsvorschlag Hintergrund oder Person? | Cross Entropy über die zwei Klassen |
| `loss_box_reg` | Wie wird die Box eines positiven Vorschlags noch genauer? | robuste Box-Regression |
| `loss_mask` | Welche Pixel innerhalb einer positiven Region gehören zur Person? | binäre Kreuzentropie über die Maskenpixel |

Die Box-Regressionen werden nur für positive Anker bzw. positive Regionsvorschläge berechnet: Für Hintergrund gibt es keine sinnvolle Zielbox. Der Mask-Head wird ebenfalls nur mit positiven Regionen trainiert. Seine rohe Ausgabe sind Masken-**Logits**; für eine sichtbare binäre Maske wenden wir später `sigmoid` implizit in `torchvision` und dann den Schwellenwert `0.5` an.

Die fünf Werte haben verschiedene Skalen. Wichtig ist deshalb vor allem ihr Verlauf und die sichtbare Qualität der Masken, nicht der direkte Vergleich zweier Teilverluste. Das Modell ist deutlich größer als die Netze in den vorigen Notebooks. Mit GPU sind die unten gewählten Epochen ein sinnvoller Kursdurchlauf; auf CPU darfst du zunächst `HEAD_EPOCHS = 1` und `FINE_TUNE_EPOCHS = 0` setzen.

In [ ]:
def move_targets_to_device(targets):
    return [{name: value.to(DEVICE) for name, value in target.items()}
            for target in targets]


LOSS_NAMES = ['loss_objectness', 'loss_rpn_box_reg', 'loss_classifier',
              'loss_box_reg', 'loss_mask']


def train_one_epoch(model, loader, optimizer):
    # Nur im train-Modus berechnet Mask R-CNN seine fünf Loss-Terme.
    model.train()
    total_loss = 0.0
    component_totals = {name: 0.0 for name in LOSS_NAMES}
    for images, targets in loader:
        images = [image.to(DEVICE) for image in images]
        targets = move_targets_to_device(targets)

        # Im Evaluation-Modus gäbe es hier Vorhersagen. Mit Targets im train-Modus
        # liefert torchvision stattdessen das Loss-Wörterbuch aus der Tabelle oben.
        losses = model(images, targets)
        loss = sum(losses.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batch_size = len(images)
        total_loss += loss.item() * batch_size
        for name, value in losses.items():
            component_totals[name] += value.item() * batch_size

    mean_components = {name: component_totals[name] / len(loader.dataset)
                       for name in LOSS_NAMES}
    return total_loss / len(loader.dataset), mean_components


@torch.no_grad()
def mean_best_instance_iou(model, loader, score_threshold=0.5):
    """Für jede echte Person: IoU mit der am besten passenden Vorhersagemaske.

    IoU = Schnittmenge / Vereinigung. Das ist eine anschauliche Kursmetrik,
    aber nicht die vollständige COCO-mAP-Auswertung mit mehreren IoU-Schwellen.
    """
    model.eval()
    scores = []
    for images, targets in loader:
        predictions = model([images[0].to(DEVICE)])[0]
        # Niedrig bewertete Vorschläge zählen nicht als fertige Detektion.
        keep = predictions['scores'].cpu() >= score_threshold
        predicted_masks = predictions['masks'].cpu()[keep, 0] >= 0.5
        true_masks = targets[0]['masks'].bool()
        for true_mask in true_masks:
            if len(predicted_masks) == 0:
                scores.append(0.0)
                continue
            # Jede echte Instanz wird mit allen Vorhersagen verglichen und erhält
            # ihren besten Partner. Das ist bei mehreren Personen pro Bild nötig.
            intersection = (predicted_masks & true_mask).sum(dim=(1, 2)).float()
            union = (predicted_masks | true_mask).sum(dim=(1, 2)).float()
            scores.append((intersection / union.clamp_min(1)).max().item())
    return float(np.mean(scores))


history = []

def run_phase(epochs, optimizer, phase_name):
    for epoch in range(1, epochs + 1):
        loss, components = train_one_epoch(model, train_loader, optimizer)
        iou = mean_best_instance_iou(model, valid_loader)
        record = {'phase': phase_name, 'epoch': epoch, 'loss': loss, 'instance_iou': iou}
        record.update(components)
        history.append(record)
        component_text = ', '.join(f'{name}={components[name]:.3f}' for name in LOSS_NAMES)
        print(f'{phase_name}, Epoche {epoch:02d}: Gesamt={loss:.3f}, IoU={iou:.3f}')
        print('  ' + component_text)

## 1. Warm-up: nur neue Köpfe trainieren

Die Bildmerkmale des vortrainierten Backbones bleiben erst konstant. Box- und Mask-Head lernen, diese Merkmale für die eine Klasse `Person` und die Penn-Fudan-Masken zu verwenden.

Das ist besonders bei einem kleinen Datensatz stabil: Die zufällig initialisierten neuen Ausgabeschichten passen sich an feste, bereits sinnvolle Merkmale an. RPN und die übrigen vortrainierten Teile bleiben im Warm-up ebenfalls eingefroren.

In [ ]:
HEAD_EPOCHS = 3
head_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
head_optimizer = torch.optim.SGD(head_parameters, lr=0.005, momentum=0.9, weight_decay=0.0005)
run_phase(HEAD_EPOCHS, head_optimizer, 'Warm-up')

## 2. Fine-Tuning: auch die Bildmerkmale leicht anpassen

Nun darf der komplette Backbone mit einer kleineren Lernrate mitlernen. Das kann die Konturen verbessern, weil die Merkmale sich etwas an die Personen in Straßen- und Campus-Szenen anpassen. Die kleinere Lernrate schützt bereits nützliche COCO-Merkmale vor zu großen Änderungen.

Der Optimierer ist in beiden Phasen SGD mit Momentum: Der Momentum-Term glättet die Richtung über mehrere Batches, `weight_decay` bremst sehr große Gewichte. Für das Fine-Tuning wird der Optimierer bewusst neu erzeugt, damit seine Lernrate eindeutig die kleinere Fine-Tuning-Lernrate ist.

In [ ]:
FINE_TUNE_EPOCHS = 3
for parameter in model.parameters():
    parameter.requires_grad = True

fine_tune_optimizer = torch.optim.SGD(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=0.0005, momentum=0.9, weight_decay=0.0005,
)
run_phase(FINE_TUNE_EPOCHS, fine_tune_optimizer, 'Fine-Tuning')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
epochs = np.arange(1, len(history) + 1)
axes[0].plot(epochs, [row['loss'] for row in history], marker='o')
axes[0].set(xlabel='Epoche', ylabel='Trainings-Loss', title='Die Trainingsverluste')
for name in LOSS_NAMES:
    axes[1].plot(epochs, [row[name] for row in history], marker='o', label=name.replace('loss_', ''))
axes[1].set(xlabel='Epoche', ylabel='mittlerer Teil-Loss', title='Die fünf Loss-Bausteine')
axes[1].legend(fontsize=8)
axes[2].plot(epochs, [row['instance_iou'] for row in history], marker='o', color='teal')
axes[2].set(xlabel='Epoche', ylabel='beste Masken-IoU', ylim=(0, 1),
            title='Überlappung echter und vorhergesagter Instanzen')
for axis in axes:
    axis.grid(alpha=0.3)
plt.tight_layout()

## Vorhersage lesen: Box, Score und eigene Pixelmaske

Im Ausgabewörterbuch steht für jede gefundene Instanz eine Box, ein Konfidenz-Score und eine Maske mit Wahrscheinlichkeiten. Der Schwellenwert `SCORE_THRESHOLD` entscheidet, welche Boxen angezeigt werden. Die Maskenwahrscheinlichkeit von mindestens `0.5` markieren wir als Personenpixel. Ein höherer Score-Schwellenwert reduziert Fehlalarme, kann aber unsichere echte Personen ausblenden.

Für die Kurve oben verwenden wir die **Instanz-IoU**. Für eine echte Maske $M$ und eine vorhergesagte Maske $P$ gilt $IoU(M,P)=\frac{|M \cap P|}{|M \cup P|}$. Jede echte Person wird mit ihrer am besten passenden, ausreichend sicheren Vorhersage verglichen. `1.0` bedeutet perfekte Überlappung, `0.0` keine Überlappung. Das ist bewusst einfacher als die offizielle COCO-mAP-Metrik und gut geeignet, um dieses kleine Kursbeispiel zu verstehen.

In [ ]:
SCORE_THRESHOLD = 0.60
MASK_THRESHOLD = 0.50

image, target = valid_dataset[0]
# Im eval-Modus erwartet das Modell keine Targets und gibt die fertigen Instanzen aus.
model.eval()
with torch.no_grad():
    prediction = model([image.to(DEVICE)])[0]

# `scores` filtert ganze Instanzen; `masks` filtert anschließend Pixel innerhalb
# jeder behaltenen Instanz. Das sind zwei verschiedene Schwellenwerte.
keep = prediction['scores'].cpu() >= SCORE_THRESHOLD
boxes = prediction['boxes'].cpu()[keep]
scores = prediction['scores'].cpu()[keep]
masks = prediction['masks'].cpu()[keep, 0] >= MASK_THRESHOLD

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
axes[0].imshow(image.permute(1, 2, 0))
axes[0].set_title('Eingabebild')

axes[1].imshow(image.permute(1, 2, 0))
for person_index, mask in enumerate(target['masks']):
    axes[1].contour(mask.numpy(), levels=[0.5], colors=[plt.cm.tab10(person_index)], linewidths=2)
axes[1].set_title('Echte Instanzmasken')

axes[2].imshow(image.permute(1, 2, 0))
for person_index, (box, score, mask) in enumerate(zip(boxes, scores, masks)):
    color = plt.cm.tab10(person_index)
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[mask.numpy()] = color
    overlay[..., 3] *= 0.42
    axes[2].imshow(overlay)
    x0, y0, x1, y1 = box.tolist()
    axes[2].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                      fill=False, edgecolor=color, linewidth=2))
    axes[2].text(x0, max(4, y0 - 4), f'Person {score:.2f}', color=color,
                 bbox={'facecolor': 'black', 'alpha': 0.55, 'pad': 1})
axes[2].set_title(f'Vorhersage: {len(boxes)} Instanz(en)')

for axis in axes:
    axis.axis('off')
plt.tight_layout()

print('Ausgegebene Scores:', [round(score.item(), 3) for score in scores])

## Modell speichern

Gespeichert werden Gewichte und die wichtigsten Modellannahmen. Beim späteren Laden muss dieselbe Mask-R-CNN-Architektur mit zwei Klassen erstellt werden, bevor der `state_dict` geladen wird.

In [ ]:
MODEL_PATH = Path('saved_models/tag_9_mask_rcnn_pennfudan.pth')
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'num_classes': NUM_CLASSES,
    'score_threshold': SCORE_THRESHOLD,
    'mask_threshold': MASK_THRESHOLD,
}, MODEL_PATH)
print('Gespeichert:', MODEL_PATH.resolve())

## Zusammenfassung

- **Instanzsegmentierung** gibt pro Person eine Klasse, eine Bounding Box und eine eigene Pixelmaske aus.
- Die bereits von Notebook 00 heruntergeladenen Penn-Fudan-Originalmasken liefern genau diese getrennten Instanzen; es ist kein neuer Datensatz nötig.
- `Mask R-CNN` kombiniert ResNet-50, FPN, RPN, RoI Align sowie getrennte Box- und Mask-Heads. Die FPN hilft dabei, große und kleine Personen zu behandeln.
- Der Gesamt-Loss addiert fünf Aufgaben: Objekt-Anker, Ankerbox, Klassenentscheidung, verfeinerte Box und Pixelmaske. Boxen und Masken lernen nur aus positiven Regionen.
- Vortrainierte COCO-Gewichte liefern gute allgemeine Bildmerkmale. Neue Ausgabeköpfe und vorsichtiges Fine-Tuning passen sie an die eine Zielklasse `Person` an.
- Die Masken-IoU misst, ob die vorhergesagte Personenfläche mit der echten Instanzfläche überlappt. Kontrolliere Scores, Boxes und Maskenüberlagerungen trotzdem immer visuell.